In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!pip -q install sentence-transformers

In [3]:

import json
import re
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer
from google.colab import drive


# Main project directory
PROJECT_DIR = Path("/content/drive/MyDrive/NLP Project")

# Manually created article metadata
INVENTORY_FILE = PROJECT_DIR / "corpus_inventory.csv"

# Four chunking strategies
STRATEGIES = [
    "fixed_size",
    "sentence_based",
    "recursive",
    "section_aware"
]

# Locate saved chunk files
possible_dirs = [
    PROJECT_DIR / "data/chunks/bge_small",
    PROJECT_DIR / "data/chunks"
]

CHUNK_DIR = next(
    (
        folder for folder in possible_dirs
        if all(
            (folder / f"chunks_{s}.jsonl").exists()
            for s in STRATEGIES
        )
    ),
    None
)

if CHUNK_DIR is None:
    raise FileNotFoundError(
        "Could not locate all four chunk JSONL files."
    )

# Output directory
EMBED_DIR = PROJECT_DIR / "data/embeddings/bge_small"
EMBED_DIR.mkdir(parents=True, exist_ok=True)

# Embedding configuration
MODEL_NAME = "BAAI/bge-small-en-v1.5"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

assert INVENTORY_FILE.exists(), "Metacorpus not found."

print("Chunk directory:", CHUNK_DIR)
print("Embedding directory:", EMBED_DIR)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)


Chunk directory: /content/drive/MyDrive/NLP Project/data/chunks/bge_small
Embedding directory: /content/drive/MyDrive/NLP Project/data/embeddings/bge_small
Model: BAAI/bge-small-en-v1.5
Device: cuda


In [4]:

# Load the manually created metacorpus
inventory = pd.read_csv(INVENTORY_FILE)

inventory.columns = inventory.columns.str.strip()
inventory["article_id"] = (
    inventory["article_id"].astype(str).str.strip()
)

# Verify unique article IDs
assert inventory["article_id"].is_unique

# Normalize DOI identifiers for duplicate detection
def normalize_doi(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    # Remove DOI URL prefixes
    value = re.sub(
        r"^(https?://(?:dx\.)?doi\.org/|doi:\s*)",
        "",
        value
    ).rstrip(".,;)")

    # Only use genuine DOI patterns for deduplication
    if re.match(r"^10\.\d{4,9}/\S+$", value):
        return value

    return None


inventory["normalized_doi"] = (
    inventory["doi_or_url"].apply(normalize_doi)
)

# Find repeated nonempty DOIs
duplicate_mask = (
    inventory["normalized_doi"].notna()
    & inventory["normalized_doi"].duplicated(keep="first")
)

duplicate_records = inventory[duplicate_mask].copy()

print("Original metadata records:", len(inventory))
print("Repeated DOI records:", len(duplicate_records))

# Save an auditable duplicate report
duplicate_records.to_csv(
    EMBED_DIR / "excluded_duplicate_dois.csv",
    index=False
)

# Keep the first article record for each DOI
inventory = inventory[~duplicate_mask].copy()

print("Unique metadata records:", len(inventory))

# Select bibliographic fields from the metacorpus
metadata_columns = [
    "article_id",
    "title",
    "author",
    "authors",
    "year",
    "venue",
    "doi_or_url",
    "topical_category",
    "source",
    "license_or_access"
]

metadata_columns = [
    col for col in metadata_columns
    if col in inventory.columns
]

manual_metadata = inventory[metadata_columns].copy()

# Article IDs allowed in the final embedding corpus
valid_article_ids = set(inventory["article_id"])

print("Metadata prepared successfully.")


Original metadata records: 201
Repeated DOI records: 20
Unique metadata records: 181
Metadata prepared successfully.


Load the BGE embedding model

In [5]:

# Load the pretrained embedding model
model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

# Get model specifications
EMBEDDING_DIM = model.get_sentence_embedding_dimension()
MODEL_MAX_TOKENS = model.max_seq_length

print("Model loaded successfully.")
print("Model:", MODEL_NAME)
print("Embedding dimension:", EMBEDDING_DIM)
print("Maximum sequence length:", MODEL_MAX_TOKENS)
print("Device:", DEVICE)

assert EMBEDDING_DIM == 384
assert MODEL_MAX_TOKENS >= 500


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Model: BAAI/bge-small-en-v1.5
Embedding dimension: 384
Maximum sequence length: 512
Device: cuda


/tmp/ipykernel_1089/3817229698.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = model.get_sentence_embedding_dimension()


Generate embeddings for all four strategies

In [6]:

# GENERATE EMBEDDINGS FOR ALL FOUR CHUNKING STRATEGIES

results = []
reference_articles = None

for strategy in STRATEGIES:

    print("\n" + "=" * 60)
    print("Processing:", strategy.upper())
    print("=" * 60)

    # 1. Load saved chunks

    chunk_file = CHUNK_DIR / f"chunks_{strategy}.jsonl"

    df = pd.read_json(chunk_file, lines=True)

    required = {
        "chunk_id", "article_id", "text",
        "page_numbers", "chunking_strategy"
    }

    if not required.issubset(df.columns):
        raise ValueError(
            f"{strategy}: missing required chunk fields"
        )

    # Validate source references and IDs
    assert df["chunk_id"].is_unique
    assert df["text"].map(
        lambda x: isinstance(x, str) and bool(x.strip())
    ).all()
    assert df["page_numbers"].map(
        lambda x: isinstance(x, list) and len(x) > 0
    ).all()
    assert df["chunking_strategy"].eq(strategy).all()

    # Check source article IDs against original metacorpus
    original_ids = set(df["article_id"])

    if not original_ids.issubset(
        set(pd.read_csv(INVENTORY_FILE)["article_id"])
    ):
        raise ValueError(
            f"{strategy}: article IDs missing from metacorpus"
        )

    # Exclude identified duplicate articles
    df = df[
        df["article_id"].isin(valid_article_ids)
    ].copy()

    # Ensure fair comparisons across strategies
    current_articles = set(df["article_id"])

    if reference_articles is None:
        reference_articles = current_articles
    elif current_articles != reference_articles:
        raise ValueError(
            "Article coverage differs between strategies."
        )

    # 2. Merge manually curated metadata

    # Replace existing bibliographic fields with
    # the manually curated metacorpus values.
    overlapping = [
        col for col in manual_metadata.columns
        if col != "article_id" and col in df.columns
    ]

    df = df.drop(columns=overlapping)

    df = df.merge(
        manual_metadata,
        on="article_id",
        how="left",
        validate="many_to_one",
        sort=False
    )

    df = df.reset_index(drop=True)

    if df.empty:
        raise ValueError(f"{strategy}: no eligible chunks")

    # 3. Validate actual model token lengths

    texts = df["text"].tolist()
    token_lengths = []

    # Check tokens in small batches to avoid large allocations
    for start in range(0, len(texts), 128):

        batch = texts[start:start + 128]

        encoded = model.tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False
        )

        token_lengths.extend(
            len(ids) for ids in encoded["input_ids"]
        )

    df["model_token_count"] = token_lengths

    if max(token_lengths) > MODEL_MAX_TOKENS:
        raise ValueError(
            f"{strategy}: oversized chunks detected."
        )

    print("Chunks:", len(df))
    print("Articles:", df["article_id"].nunique())
    print("Maximum tokens:", max(token_lengths))

    # 4. Generate embeddings

    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    # Validate embedding matrix
    assert embeddings.shape == (
        len(df), EMBEDDING_DIM
    )

    assert np.isfinite(embeddings).all()

    # Normalized embeddings should have unit length
    assert np.allclose(
        np.linalg.norm(embeddings, axis=1),
        1.0,
        atol=1e-3
    )

    # 5. Preserve vector-to-metadata alignment

    df.insert(
        0,
        "vector_id",
        np.arange(len(df), dtype=np.int64)
    )

    # 6. Save embeddings and metadata

    output_dir = EMBED_DIR / strategy
    output_dir.mkdir(parents=True, exist_ok=True)

    np.save(
        output_dir / "embeddings.npy",
        embeddings
    )

    df.to_json(
        output_dir / "metadata.jsonl",
        orient="records",
        lines=True,
        force_ascii=False
    )

    # Save experiment configuration
    config = {
        "model_name": MODEL_NAME,
        "chunking_strategy": strategy,
        "chunk_size": 500,
        "chunk_overlap": 80,
        "embedding_dimension": EMBEDDING_DIM,
        "model_max_tokens": MODEL_MAX_TOKENS,
        "batch_size": BATCH_SIZE,
        "normalized": True,
        "total_chunks": len(df),
        "total_articles": df["article_id"].nunique()
    }

    with open(
        output_dir / "config.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(config, file, indent=4)

    results.append({
        "Strategy": strategy,
        "Chunks": len(df),
        "Articles": df["article_id"].nunique(),
        "Dimensions": embeddings.shape[1],
        "Max Tokens": max(token_lengths),
        "Status": "Saved"
    })

    print("Embedding shape:", embeddings.shape)
    print("Saved to:", output_dir)

    # Free memory before the next strategy
    del embeddings, df, texts

# Display final summary
results_df = pd.DataFrame(results)

print("\nEMBEDDING GENERATION COMPLETED")
display(results_df)



Processing: FIXED_SIZE
Chunks: 5703
Articles: 180
Maximum tokens: 504


Batches:   0%|          | 0/179 [00:00<?, ?it/s]

Embedding shape: (5703, 384)
Saved to: /content/drive/MyDrive/NLP Project/data/embeddings/bge_small/fixed_size

Processing: SENTENCE_BASED
Chunks: 5839
Articles: 180
Maximum tokens: 502


Batches:   0%|          | 0/183 [00:00<?, ?it/s]

Embedding shape: (5839, 384)
Saved to: /content/drive/MyDrive/NLP Project/data/embeddings/bge_small/sentence_based

Processing: RECURSIVE
Chunks: 6106
Articles: 180
Maximum tokens: 502


Batches:   0%|          | 0/191 [00:00<?, ?it/s]

Embedding shape: (6106, 384)
Saved to: /content/drive/MyDrive/NLP Project/data/embeddings/bge_small/recursive

Processing: SECTION_AWARE
Chunks: 7623
Articles: 180
Maximum tokens: 502


Batches:   0%|          | 0/239 [00:00<?, ?it/s]

Embedding shape: (7623, 384)
Saved to: /content/drive/MyDrive/NLP Project/data/embeddings/bge_small/section_aware

EMBEDDING GENERATION COMPLETED


,Strategy,Chunks,Articles,Dimensions,Max Tokens,Status
0,fixed_size,5703,180,384,504,Saved
1,sentence_based,5839,180,384,502,Saved
2,recursive,6106,180,384,502,Saved
3,section_aware,7623,180,384,502,Saved


Verify saved embedding files

In [7]:

# VERIFY SAVED EMBEDDINGS AND METADATA

for strategy in STRATEGIES:

    folder = EMBED_DIR / strategy

    # Load saved embeddings without copying them into RAM
    embeddings = np.load(
        folder / "embeddings.npy",
        mmap_mode="r"
    )

    metadata = pd.read_json(
        folder / "metadata.jsonl",
        lines=True
    )

    # Verify embedding/metadata alignment
    assert len(embeddings) == len(metadata)
    assert embeddings.shape[1] == EMBEDDING_DIM

    assert metadata["chunk_id"].is_unique

    assert np.array_equal(
        metadata["vector_id"].to_numpy(),
        np.arange(len(metadata))
    )

    assert metadata["page_numbers"].map(bool).all()

    print("\nStrategy:", strategy)
    print("Embedding shape:", embeddings.shape)
    print("Metadata rows:", len(metadata))
    print("Unique articles:", metadata["article_id"].nunique())
    print("Validation: PASSED")

print("\nAll four embedding datasets verified successfully.")



Strategy: fixed_size
Embedding shape: (5703, 384)
Metadata rows: 5703
Unique articles: 180
Validation: PASSED

Strategy: sentence_based
Embedding shape: (5839, 384)
Metadata rows: 5839
Unique articles: 180
Validation: PASSED

Strategy: recursive
Embedding shape: (6106, 384)
Metadata rows: 6106
Unique articles: 180
Validation: PASSED

Strategy: section_aware
Embedding shape: (7623, 384)
Metadata rows: 7623
Unique articles: 180
Validation: PASSED

All four embedding datasets verified successfully.
